In [5]:
from pyspark.sql.types import *

# create the schema for the table

orderSchema = StructType([
    StructField("SalesOrderNumber", StringType()),
    StructField("SalesOrderLineNumber", IntegerType()),
    StructField("OrderDate", DateType()),
    StructField("CustomerName", StringType()),
    StructField("Email", StringType()),
    StructField("Item", StringType()),
    StructField("Quantity", IntegerType()),
    StructField("UnitPrice", FloatType()),
    StructField("Tax", FloatType())
    ])

# import all files from bronze folder of lakehouse

df = spark.read.format("csv").option("header", "false").schema(orderSchema).load("Files/bronze/*.csv")

# Display the first 10 rows of the dataframe to preview the data

display(df.limit(10))

StatementMeta(, 8693686b-eb5d-4a91-868b-5b3c6c336939, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 133349b8-7902-491b-b91e-7835b139c262)

In [6]:
# data validation and cleanup

# imports the necessary functions from PySpark
from pyspark.sql.functions import when, lit, col, current_timestamp, input_file_name

''' Add columns IsFlagged, CreatedTS and ModifiedTS (new columns to track the source file name, 
whether the order was flagged as being a before the fiscal year of interest,
when the row was created and modified.'''

df = df.withColumn("FileName", input_file_name()) \
        .withColumn("Isflagged", when(col("OrderDate") < "2019-08-01", True).otherwise(False)) \
        .withColumn("CreatedTS", current_timestamp()) \
        .withColumn("ModifiedTS", current_timestamp())

# Update CustomerName to "Unknown" if CustomerName NULL or Empty (# lit() creates column of constant/literal value)

df = df.withColumn("CustomerName", when((col("CustomerName").isNull() | (col("CustomerName") == "")), lit("Unknown")) \
.otherwise(col("CustomerName")))

StatementMeta(, 8693686b-eb5d-4a91-868b-5b3c6c336939, 8, Finished, Available, Finished, False)

In [12]:
spark.sql("SHOW CATALOGS").show()
spark.sql("SHOW SCHEMAS").show()
spark.sql("SELECT current_catalog(), current_database()").show()


StatementMeta(, 9410e246-5c7a-486a-a7bb-55567780ccd4, 14, Finished, Available, Finished, False)

+-------------+
|      catalog|
+-------------+
|spark_catalog|
+-------------+

+----------+
| namespace|
+----------+
|SalesHouse|
+----------+

+-----------------+------------------+
|current_catalog()|current_database()|
+-----------------+------------------+
|    spark_catalog|        saleshouse|
+-----------------+------------------+



In [3]:
# Define the schema for the sales_silver table
    
from pyspark.sql.types import *
from delta.tables import *
    
DeltaTable.createIfNotExists(spark) \
    .tableName("saleshouse.sales_silver") \
    .addColumn("SalesOrderNumber", StringType()) \
    .addColumn("SalesOrderLineNumber", IntegerType()) \
    .addColumn("OrderDate", DateType()) \
    .addColumn("CustomerName", StringType()) \
    .addColumn("Email", StringType()) \
    .addColumn("Item", StringType()) \
    .addColumn("Quantity", IntegerType()) \
    .addColumn("UnitPrice", FloatType()) \
    .addColumn("Tax", FloatType()) \
    .addColumn("FileName", StringType()) \
    .addColumn("IsFlagged", BooleanType()) \
    .addColumn("CreatedTS", DateType()) \
    .addColumn("ModifiedTS", DateType()) \
    .execute()

StatementMeta(, 8693686b-eb5d-4a91-868b-5b3c6c336939, 5, Finished, Available, Finished, False)

In [7]:
'''Update existing records and insert new ones based on a condition defined by the columns 
SalesOrderNumber, OrderDate, CustomerName, and Item.'''
# Upsert (Update + Insert)

from delta.tables import *

deltaTable = DeltaTable.forPath(spark, "Tables/sales_silver")

dfUpdates = df

deltaTable.alias("silver") \
.merge(
    dfUpdates.alias("updates"),
    "silver.SalesOrderNumber = updates.SalesOrderNumber and silver.OrderDate = updates.OrderDate and silver.CustomerName = updates.CustomerName and silver.Item = updates.Item"
)\
.whenMatchedUpdate(set = 
{

}
)\
.whenNotMatchedInsert(values =
{
    "SalesOrderNumber": "updates.SalesOrderNumber",
    "SalesOrderLineNumber": "updates.SalesOrderLineNumber",
    "OrderDate": "updates.OrderDate",
    "CustomerName": "updates.CustomerName",
    "Email": "updates.Email",
    "Item": "updates.Email",
    "Quantity":"updates.Quantity",
    "UnitPrice": "updates.UnitPrice",
    "Tax": "updates.Tax",
    "FileName": "updates.FileName",
    "IsFlagged": "updates.IsFlagged",
    "CreatedTS": "updates.CreatedTS",
    "ModifiedTS": "updates.ModifiedTS"
}
)\
.execute()

'''This operation is important because it enables you to 
update existing records in the table based on the values of specific columns,
and insert new records when no match is found. 
This is a common requirement when you’re loading data from 
a source system that may contain updates to existing and new records.'''

StatementMeta(, 8693686b-eb5d-4a91-868b-5b3c6c336939, 9, Finished, Available, Finished, False)

'This operation is important because it enables you to \nupdate existing records in the table based on the values of specific columns,\nand insert new records when no match is found. \nThis is a common requirement when you’re loading data from \na source system that may contain updates to existing and new records.'